# Notebook B — Marqueurs RSWA / EMG : Boxplots et statistiques

**Input :** `results/emg/rbd_emg_events_and_summary_4s_per_channel.csv` (sortie de `03_segment_rswa.py`)  
**Input :** `data/patients_label.txt`  
**Output :** figures dans `results/emg/boxplots/` + tables stats dans `results/emg/csv/`

Ce notebook produit :
1. Boxplots des 6 métriques RSWA par canal EMG × groupe clinique
2. Tests statistiques (Kruskal–Wallis + Mann–Whitney Holm)
3. Export Excel récapitulatif avec étoiles

In [ ]:
# ============================================================
# PARAMÈTRES  ←  À MODIFIER
# ============================================================
EMG_CSV     = "results/emg/rbd_emg_events_and_summary_4s_per_channel.csv"
LABELS_TXT  = "data/patients_label.txt"
OUTPUT_FIGS = "results/emg/boxplots"
OUTPUT_STATS= "results/emg/csv"

SAVE_FIGS  = True
FIG_FORMAT = "pdf"
DPI        = 300
ALPHA      = 0.05

PALETTE = {
    "EAI":   "#C9D175",
    "Narco": "#F15854",
    "SYN":   "#44AA99",
    "TCSPi": "#BEBEBE",
}
GROUP_ORDER = ["SYN", "Narco", "TCSPi", "EAI"]

# Métriques à visualiser
METRICS = {
    "pct_ep_with_phasic":   "% REM episodes with ≥1 phasic event",
    "mean_phasic_density":  "Phasic density (events/min)",
    "mean_perc_phasic":     "% REM time as phasic activity",
    "pct_heavy":            "% REM episodes with >75% phasic",
    "median_tonic_ratio":   "Tonic ratio (REM/NREM)",
    "pct_rswa":             "% REM epochs with RSWA",
}

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from itertools import combinations

import scipy.stats as st
from statsmodels.stats.multitest import multipletests

Path(OUTPUT_FIGS).mkdir(parents=True, exist_ok=True)
Path(OUTPUT_STATS).mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="talk")
matplotlib.rcParams.update({"axes.spines.top": False, "axes.spines.right": False})
print("Librairies chargées.")

In [ ]:
# ============================================================
# CHARGEMENT ET FUSION
# ============================================================
df_raw = pd.read_csv(EMG_CSV)
df_raw["patient_id"] = df_raw["patient_id"].astype(str).str.strip()

# Labels
try:
    lbl = pd.read_csv(LABELS_TXT)
    cols = {c.lower(): c for c in lbl.columns}
    id_col  = cols.get("patient_id") or cols.get("identifiant")
    lbl_col = cols.get("label_str") or cols.get("diagnostic") or cols.get("label")
    lbl = lbl[[id_col, lbl_col]].rename(columns={id_col: "patient_id", lbl_col: "group"})
except Exception:
    lbl = pd.read_csv(LABELS_TXT, header=None, names=["patient_id", "group"])

lbl["patient_id"] = lbl["patient_id"].astype(str).str.strip()

def to_macro(s):
    u = str(s).upper()
    if any(k in u for k in ("PARK","MPI","AMS","DCL","DLB","PAF")): return "SYN"
    if "NARCO" in u: return "Narco"
    if "TCSP" in u or "RBDI" in u: return "TCSPi"
    if "EAI" in u or "ENCEPHALITE" in u: return "EAI"
    return s

lbl["group"] = lbl["group"].map(to_macro)
df_raw = df_raw.merge(lbl, on="patient_id", how="left")

print(f"Groupes : {df_raw['group'].value_counts().to_dict()}")
print(f"Canaux  : {sorted(df_raw['channel'].dropna().unique())}")

In [ ]:
# ============================================================
# CALCUL DES MÉTRIQUES PATIENT-LEVEL
# ============================================================
epoch_len = df_raw["epoch_len_sec"].dropna().iloc[0]
ep = df_raw[df_raw["type"] == "REM_EPOCH_4S"].copy()

# % REM avec ≥1 phasic
m1 = (ep.groupby(["patient_id","group","channel","episode_index"])
        .agg(has_phasic=("phasic_count", lambda x: (x>0).any()))
        .reset_index()
        .groupby(["patient_id","group","channel"])
        .agg(pct_ep_with_phasic=("has_phasic","mean"))
        .reset_index())
m1["pct_ep_with_phasic"] *= 100

# Densité phasique (nb/min)
m2 = (ep.groupby(["patient_id","group","channel","episode_index"])
        .agg(pc=("phasic_count","sum"), ne=("epoch_len_sec","count"))
        .assign(dens=lambda x: 60*x.pc/(x.ne*epoch_len))
        .groupby(["patient_id","group","channel"])
        .agg(mean_phasic_density=("dens","mean"))
        .reset_index())

# % REM en phasic
m3 = (ep.groupby(["patient_id","group","channel","episode_index"])
        .agg(pt=("phasic_time_sec","sum"), tt=("epoch_len_sec",lambda x: len(x)*epoch_len))
        .assign(perc=lambda x: x.pt/x.tt)
        .groupby(["patient_id","group","channel"])
        .agg(mean_perc_phasic=("perc","mean"))
        .reset_index())
m3["mean_perc_phasic"] *= 100

# % épisodes très phasiques
m4 = (ep.assign(heavy=lambda x: x["phasic_ratio"]>0.75)
        .groupby(["patient_id","group","channel","episode_index"])
        .agg(ep_heavy=("heavy","any"))
        .groupby(["patient_id","group","channel"])
        .agg(pct_heavy=("ep_heavy","mean"))
        .reset_index())
m4["pct_heavy"] *= 100

# Tonic ratio médian
m5 = (ep.groupby(["patient_id","group","channel","episode_index"])
        .agg(med=("tonic_ratio","median"))
        .groupby(["patient_id","group","channel"])
        .agg(median_tonic_ratio=("med","median"))
        .reset_index())

# % RSWA
m6 = (ep.assign(is_rswa=lambda x: x["tonic_ratio"]>1.3)
        .groupby(["patient_id","group","channel","episode_index"])
        .agg(rswa=("is_rswa","any"))
        .groupby(["patient_id","group","channel"])
        .agg(pct_rswa=("rswa","mean"))
        .reset_index())
m6["pct_rswa"] *= 100

# Fusion
summary = (m1
    .merge(m2, on=["patient_id","group","channel"], how="left")
    .merge(m3, on=["patient_id","group","channel"], how="left")
    .merge(m4, on=["patient_id","group","channel"], how="left")
    .merge(m5, on=["patient_id","group","channel"], how="left")
    .merge(m6, on=["patient_id","group","channel"], how="left")
)
summary = summary[summary["group"].isin(GROUP_ORDER)]

summary.to_csv(Path(OUTPUT_STATS)/"rswa_summary_patient_level.csv", index=False)
print(f"Summary : {summary.shape} | Canaux : {summary['channel'].nunique()}")

In [ ]:
# ============================================================
# HELPERS STAT ET BOXPLOT
# ============================================================
def iqr_bounds(a):
    q1, q3 = np.percentile(a, [25, 75])
    return q1 - 1.5*(q3-q1), q3 + 1.5*(q3-q1)

def p_stars(p):
    if pd.isna(p): return ""
    if p < 0.001:  return "***"
    if p < 0.01:   return "**"
    if p < 0.05:   return "*"
    return ""

def run_stats(data, vcol):
    groups = {g: sub[vcol].dropna().values
              for g, sub in data.groupby("group") if len(sub) >= 3}
    if len(groups) < 2:
        return np.nan, pd.DataFrame()
    kw_stat, kw_p = st.kruskal(*groups.values())
    pw_rows = []
    for (g1, x), (g2, y) in combinations(groups.items(), 2):
        u, p_raw = st.mannwhitneyu(x, y, alternative="two-sided")
        pw_rows.append({"group1":g1,"group2":g2,"n1":len(x),"n2":len(y),
                        "U":u,"p_raw":p_raw})
    pw = pd.DataFrame(pw_rows)
    if not pw.empty:
        _, p_holm, _, _ = multipletests(pw["p_raw"], method="holm")
        pw["p_holm"] = p_holm
        pw["stars"]  = pw["p_holm"].map(p_stars)
    return kw_p, pw

def boxplot_metric(data, channel, metric, ylabel, save_path=None):
    sub = data[data["channel"]==channel].copy()
    if sub.empty: return

    filtered = []
    for g, grp in sub.groupby("group"):
        lo, hi = iqr_bounds(grp[metric].dropna().values)
        filtered.append(grp[(grp[metric]>=lo)&(grp[metric]<=hi)])
    sub_f = pd.concat(filtered, ignore_index=True)

    kw_p, pw = run_stats(sub_f, metric)
    order = [g for g in GROUP_ORDER if g in sub_f["group"].unique()]

    fig, ax = plt.subplots(figsize=(7, 5))
    sns.boxplot(data=sub_f, x="group", y=metric, order=order,
                palette=PALETTE, showfliers=False, ax=ax, legend=False)
    sns.stripplot(data=sub_f, x="group", y=metric, order=order,
                  color="black", alpha=0.55, size=4, jitter=True, ax=ax)

    # Annotations significatives
    if not pw.empty:
        sig = pw[pw["stars"]!=""]
        ymax   = sub_f[metric].max()
        yrange = sub_f[metric].max() - sub_f[metric].min()
        step   = yrange * 0.12
        for k, (_, r) in enumerate(sig.iterrows()):
            if r["group1"] not in order or r["group2"] not in order: continue
            x1, x2 = order.index(r["group1"]), order.index(r["group2"])
            y_ = ymax + step*(k+1)
            ax.plot([x1,x1,x2,x2],[y_-step*0.2,y_,y_,y_-step*0.2],lw=1.2,color="black")
            ax.text((x1+x2)/2, y_, r["stars"], ha="center", va="bottom",
                    fontsize=13, fontweight="bold")

    kw_label = f"Kruskal–Wallis p={kw_p:.3f}" if not np.isnan(kw_p) else ""
    ax.set_title(f"{ylabel}\nCanal : {channel}\n{kw_label}", fontsize=11)
    ax.set_xlabel("")
    ax.set_ylabel(ylabel)
    plt.tight_layout()
    if save_path and SAVE_FIGS:
        fig.savefig(save_path, dpi=DPI, bbox_inches="tight")
    plt.show()
    plt.close(fig)

print("Helpers prêts.")

In [ ]:
# ============================================================
# FIGURES — 6 métriques × tous les canaux EMG
# ============================================================
channels = sorted(summary["channel"].unique())
print(f"Canaux EMG disponibles : {channels}")

for metric, ylabel in METRICS.items():
    for ch in channels:
        save_path = Path(OUTPUT_FIGS) / f"{metric}_{ch}.{FIG_FORMAT}"
        boxplot_metric(summary, channel=ch, metric=metric,
                       ylabel=ylabel, save_path=save_path)

In [ ]:
# ============================================================
# EXPORT STATISTIQUES COMPLET
# ============================================================
kw_rows, pw_rows, means_rows = [], [], []

for metric in METRICS:
    for ch in channels:
        sub = summary[summary["channel"]==ch].copy()
        if sub["group"].nunique() < 2: continue

        for g, grp in sub.groupby("group"):
            vals = grp[metric].dropna()
            means_rows.append({"metric":metric,"channel":ch,"group":g,
                                "mean":vals.mean(),"sd":vals.std(),"n":len(vals)})

        kw_p, pw = run_stats(sub, metric)
        kw_rows.append({"metric":metric,"channel":ch,"p_kw":kw_p})
        if not pw.empty:
            for _, r in pw.iterrows():
                pw_rows.append({"metric":metric,"channel":ch,**r.to_dict()})

pd.DataFrame(means_rows).to_csv(Path(OUTPUT_STATS)/"rswa_means_per_metric_channel_group.csv", index=False)
pd.DataFrame(kw_rows).to_csv(Path(OUTPUT_STATS)/"rswa_kruskal_results.csv", index=False)
pw_df = pd.DataFrame(pw_rows)
pw_df.to_csv(Path(OUTPUT_STATS)/"rswa_mannwhitney_posthoc.csv", index=False)

sig_df = pw_df[pw_df["stars"]!=""] if not pw_df.empty and "stars" in pw_df.columns else pd.DataFrame()
if not sig_df.empty:
    sig_df.to_excel(Path(OUTPUT_STATS)/"rswa_significant_comparisons.xlsx", index=False)
    print(f"{len(sig_df)} comparaisons significatives exportées.")
print("Terminé. Fichiers dans", OUTPUT_STATS)

In [ ]:
# ============================================================
# FIGURE BONUS — Heatmap phasic density × canal × groupe
# ============================================================
pivot = (summary.groupby(["group","channel"])["mean_phasic_density"]
                .mean().reset_index()
                .pivot(index="group", columns="channel", values="mean_phasic_density"))

fig, ax = plt.subplots(figsize=(max(8, len(pivot.columns)*0.9), 4))
sns.heatmap(pivot, cmap="YlOrRd", annot=True, fmt=".2f", linewidths=0.4,
            cbar_kws={"label":"Phasic density (events/min)"}, ax=ax)
ax.set_title("Densité phasique moyenne par groupe × canal", fontsize=12)
ax.set_xlabel("Canal EMG")
ax.set_ylabel("Groupe clinique")
plt.tight_layout()
if SAVE_FIGS:
    fig.savefig(Path(OUTPUT_FIGS)/f"heatmap_phasic_density.{FIG_FORMAT}",
                dpi=DPI, bbox_inches="tight")
plt.show()
plt.close(fig)